# Simple TensorFlow.js Model Training
## Moisture Detection Image Classification

This notebook trains an image classification model and exports it directly to TensorFlow.js format.

**Setup**: Use GPU runtime in Google Colab for faster training.

## 1. Setup and Mount Drive

In [ ]:
# Install required packages with dependency handling
import os
import shutil
import random
from pathlib import Path
os.system('pip install --upgrade pip -q')
os.system('pip install tensorflowjs -q')

# Import packages
import tensorflow as tf
import tensorflowjs as tfjs
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set your data path - update this to your Google Drive folder
DATA_PATH = '/content/drive/MyDrive/moisture-detection-data/'  # Update this path

# Set max images per class for training (0 = use all images)
MAX_IMAGES_PER_CLASS = 100  # Change this to limit images per class

# Verify data path exists
if os.path.exists(DATA_PATH):
    print(f"✓ Data path found: {DATA_PATH}")
    print(f"Classes found: {os.listdir(DATA_PATH)}")
    print(f"Max images per class: {MAX_IMAGES_PER_CLASS if MAX_IMAGES_PER_CLASS > 0 else 'All images'}")
else:
    print(f"✗ Data path not found: {DATA_PATH}")
    print("Please update DATA_PATH to match your Google Drive folder structure")

## 2. Data Preparation with Image Limiting

In [ ]:
# Create a limited dataset if MAX_IMAGES_PER_CLASS is set
if MAX_IMAGES_PER_CLASS > 0:
    print(f"Creating limited dataset with {MAX_IMAGES_PER_CLASS} images per class...")
    
    # Create temporary directory for limited dataset
    LIMITED_DATA_PATH = '/content/limited_data/'
    if os.path.exists(LIMITED_DATA_PATH):
        shutil.rmtree(LIMITED_DATA_PATH)
    os.makedirs(LIMITED_DATA_PATH)
    
    # Process each class folder
    for class_name in os.listdir(DATA_PATH):
        class_path = os.path.join(DATA_PATH, class_name)
        if os.path.isdir(class_path):
            # Create class directory in limited dataset
            limited_class_path = os.path.join(LIMITED_DATA_PATH, class_name)
            os.makedirs(limited_class_path)
            
            # Get all image files
            image_files = [f for f in os.listdir(class_path) 
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            
            # Randomly select images up to the limit
            selected_images = random.sample(image_files, 
                                          min(len(image_files), MAX_IMAGES_PER_CLASS))
            
            # Copy selected images
            for img_file in selected_images:
                src_path = os.path.join(class_path, img_file)
                dst_path = os.path.join(limited_class_path, img_file)
                shutil.copy2(src_path, dst_path)
            
            print(f"Class '{class_name}': {len(selected_images)} images selected")
    
    # Use limited dataset for training
    TRAINING_DATA_PATH = LIMITED_DATA_PATH
    print(f"\n✓ Limited dataset created at: {LIMITED_DATA_PATH}")
else:
    # Use full dataset
    TRAINING_DATA_PATH = DATA_PATH
    print("Using full dataset for training")

print(f"Training data path: {TRAINING_DATA_PATH}")

In [ ]:
# Data generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

# Training data
train_generator = train_datagen.flow_from_directory(
    TRAINING_DATA_PATH,
    target_size=(224, 224),
    batch_size=16,  # Smaller batch size for Colab
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Validation data
validation_generator = train_datagen.flow_from_directory(
    TRAINING_DATA_PATH,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Number of classes: {train_generator.num_classes}")
print(f"Class indices: {train_generator.class_indices}")

## 3. Model Creation

In [ ]:
# Create model with MobileNetV2 transfer learning
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model initially
base_model.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model created and compiled successfully")
print(f"Total parameters: {model.count_params():,}")

## 4. Training

In [ ]:
# Training callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=10, 
        restore_best_weights=True, 
        monitor='val_accuracy'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        patience=5, 
        factor=0.5, 
        monitor='val_accuracy'
    )
]

# Phase 1: Train with frozen base model
print("Phase 1: Training with frozen base model...")
history1 = model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"Phase 1 completed. Best validation accuracy: {max(history1.history['val_accuracy']):.4f}")

In [ ]:
# Phase 2: Fine-tuning with unfrozen base model
print("Phase 2: Fine-tuning with unfrozen base model...")

# Unfreeze base model
base_model.trainable = True

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training
history2 = model.fit(
    train_generator,
    epochs=30,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"Phase 2 completed. Best validation accuracy: {max(history2.history['val_accuracy']):.4f}")
print("Training completed successfully!")

## 5. Export to TensorFlow.js

In [ ]:
# Create output directory
output_dir = '/content/moisture_detection_model'
os.makedirs(output_dir, exist_ok=True)

# Convert to TensorFlow.js format
print("Converting model to TensorFlow.js format...")
tfjs.converters.save_keras_model(
    model,
    output_dir,
    quantization_bytes=2  # Reduce model size
)

print(f"✓ Model exported to: {output_dir}")
print("Files created:")
for file in os.listdir(output_dir):
    file_path = os.path.join(output_dir, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  - {file} ({size_mb:.2f} MB)")

## 6. Create Model Metadata

In [ ]:
import json
from datetime import datetime

# Get final validation accuracy
final_accuracy = max(history2.history['val_accuracy']) if 'val_accuracy' in history2.history else max(history1.history['val_accuracy'])

# Create metadata
metadata = {
    "modelName": "moisture-detection-custom",
    "version": "1.0.0",
    "description": "Custom trained moisture detection model using MobileNetV2 transfer learning",
    "trainingDate": datetime.now().strftime("%Y-%m-%d"),
    "modelType": "Image Classification",
    "architecture": "MobileNetV2 + Custom Head",
    "inputShape": [224, 224, 3],
    "classes": list(train_generator.class_indices.keys()),
    "classIndices": train_generator.class_indices,
    "performance": {
        "validationAccuracy": float(final_accuracy),
        "trainingSamples": train_generator.samples,
        "validationSamples": validation_generator.samples
    },
    "preprocessing": {
        "rescale": "1/255",
        "targetSize": [224, 224]
    },
    "trainingConfig": {
        "maxImagesPerClass": MAX_IMAGES_PER_CLASS if MAX_IMAGES_PER_CLASS > 0 else "all"
    }
}

# Save metadata
metadata_path = os.path.join(output_dir, 'metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("✓ Metadata created")
print(f"Final validation accuracy: {final_accuracy:.4f}")
print(f"Classes: {list(train_generator.class_indices.keys())}")
print(f"Images per class limit: {MAX_IMAGES_PER_CLASS if MAX_IMAGES_PER_CLASS > 0 else 'No limit'}")

## 7. Download Model Files

In [ ]:
# Create zip file for download
import zipfile

zip_path = '/content/moisture_detection_model.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, output_dir)
            zipf.write(file_path, arcname)

print(f"✓ Model files zipped: {zip_path}")

# Download the zip file
from google.colab import files
files.download(zip_path)

print("\n🎉 Training completed successfully!")
print("\nNext steps:")
print("1. Extract the downloaded zip file")
print("2. Upload model.json and .bin files to your web server")
print("3. Update your application to use the new model")
print(f"4. Use the class indices: {train_generator.class_indices}")
print(f"\nModel trained with {MAX_IMAGES_PER_CLASS if MAX_IMAGES_PER_CLASS > 0 else 'all'} images per class")